In [1]:
import random
import math
import csv
import pandas as pd

def generate_partition_array(n, m=1000):
    """
    n: 전체 원소 개수 (s1 + s2)
    m: 개별 원소의 최대 허용값 (기본값 1000)
    A: s1을 uniform 샘플링한 뒤 그 합으로 결정됨
    """

    def break_stick(target_sum, count, max_val):
        # s2 생성: target_sum을 count개의 조각으로 나누는 로직 (Stick Breaking)
        # 원소 범위: [1, max_val]
        attempts = 0
        while attempts < 10000:
            points = sorted(random.sample(range(1, target_sum), count - 1))
            points = [0] + points + [target_sum]
            pieces = [points[i+1] - points[i] for i in range(count)]
            if all(p <= max_val for p in pieces):
                return pieces
            attempts += 1
        return None  # 실패 시 None 반환

    min_k = n // 2 + 1
    max_k = n - n // 5 - 1

    while True:
        # 1. k 결정
        # s1이 s2보다 원소 개수가 많아야 함: k > n-k  =>  k >= n//2 + 1
        # s2 원소 개수가 n/5보다 많아야 함: n-k > n/5  =>  k <= n - n//5 - 1
        k = random.randint(min_k, max_k)

        # 2. s1 생성: [250, 500] 정수 균등 분포에서 k개 독립 샘플링
        s1 = [random.randint(250, 500) for _ in range(k)]
        A = sum(s1)

        # 3. s2 생성: 원소 범위 [1, 1000], 실패 시 k부터 다시 뽑음
        s2 = break_stick(A, n - k, max_val=1000)
        if s2 is not None:
            break

    print(f"k={k}, n-k={n-k}, A={A}")
    print(f"s1 ({k}개): {s1}")
    print(f"s2 ({n-k}개): {s2}")
    print(f"s1합: {sum(s1)}")
    print(f"s2합: {sum(s2)}")

    # 4. 두 집합 병합
    instance = s1 + s2

    return instance


In [2]:
def apply_johnsons_rule_and_save(g, c1, w1, c2, w2, nnn):
    k = 1  # 수식 내 파라미터, 차피 순서 안바뀌니 1로 고정해서 계산
    jobs = []

    # 1. 수식 적용: p_ij = ((c_ij * w_ij) / k)^(k/(k+1))
    for j in range(1, g + 2):
        p1_val = (c1[j] * w1[j] / k) ** (k / (k + 1))
        p2_val = (c2[j] * w2[j] / k) ** (k / (k + 1))

        jobs.append({
            'JOB_ID': f"j{j-1}",
            'p1': p1_val,
            'p2': p2_val,
            'C1': c1[j], 'W1': w1[j],
            'C2': c2[j], 'W2': w2[j]
        })

    # 2. 기계 1과 2의 시간이 같으면 앞에 배치 (p1 <= p2)
    group1 = [j for j in jobs if j['p1'] <= j['p2']]
    group2 = [j for j in jobs if j['p1'] > j['p2']]

    # Group 1: p1 기준 오름차순
    group1.sort(key=lambda x: x['p1'])
    # Group 2: p2 기준 내림차순
    group2.sort(key=lambda x: x['p2'], reverse=True)

    sorted_jobs = group1 + group2

    # 3. CSV 저장
    filename_sorted = f"2026ADP_Uniform_{g+1}_N_{nnn:02d}_inst_input.csv"
    df = pd.DataFrame(sorted_jobs)
    final_df = df[['JOB_ID', 'C1', 'W1', 'C2', 'W2']]
    final_df.columns = ['JOB_ID', 'JOB_C1', 'JOB_W1', 'JOB_C2', 'JOB_W2']

    final_df.to_csv(filename_sorted, index=False)
    print(f"재배열 완료 (p1=p2 우선배치 적용): {filename_sorted}")


In [7]:
def transform_to_micp_instance(inputn, nnn):
    g = inputn

    # c1, c2, w1, w2: 모든 잡(j=1~g+1)에 대해 [250,500] uniform 독립 샘플링
    c1 = [0.0] * (g + 2)
    c2 = [0.0] * (g + 2)
    w1 = [0.0] * (g + 2)
    w2 = [0.0] * (g + 2)

    for j in range(1, g + 2):
        c1[j] = random.randint(250, 500)
        c2[j] = random.randint(250, 500)
        w1[j] = random.randint(250, 500)
        w2[j] = random.randint(250, 500)

    filename = f"2026ADP_Uniform_saved_{g+1}_N_{nnn:02d}_inst.csv"
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['JOB_ID', 'JOB_C1', 'JOB_W1', 'JOB_C2', 'JOB_W2'])
        for j in range(1, g + 2):
            job_id = f"j{j-1}"
            writer.writerow([job_id, c1[j], w1[j], c2[j], w2[j]])

    print(f"CSV 파일 저장 완료: {filename}")

    apply_johnsons_rule_and_save(g, c1, w1, c2, w2, nnn)


In [ ]:
NNN = [300,400,500]
inputm = 1000

for N in NNN:
    inputn = N - 1
    for nnn in range(1, 31):
        #dd = generate_partition_array(n=inputn, m=inputm)
        transform_to_micp_instance(inputn, nnn=nnn)


CSV 파일 저장 완료: 2026ADP_Uniform_saved_200_N_11_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Uniform_200_N_11_inst_input.csv
CSV 파일 저장 완료: 2026ADP_Uniform_saved_200_N_12_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Uniform_200_N_12_inst_input.csv
CSV 파일 저장 완료: 2026ADP_Uniform_saved_200_N_13_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Uniform_200_N_13_inst_input.csv
CSV 파일 저장 완료: 2026ADP_Uniform_saved_200_N_14_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Uniform_200_N_14_inst_input.csv
CSV 파일 저장 완료: 2026ADP_Uniform_saved_200_N_15_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Uniform_200_N_15_inst_input.csv
CSV 파일 저장 완료: 2026ADP_Uniform_saved_200_N_16_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Uniform_200_N_16_inst_input.csv
CSV 파일 저장 완료: 2026ADP_Uniform_saved_200_N_17_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Uniform_200_N_17_inst_input.csv
CSV 파일 저장 완료: 2026ADP_Uniform_saved_200_N_18_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Uniform_200_N_18_inst_input.csv
CSV 파일 저장 완료: 2026ADP_Uniform_saved_200_N_19_inst.csv
재배